# Plant-to-Watershed AI Lab — Google Colab Training Notebook

**Research Context:**  
Coupling Individual Plant Models with SWAT Hydrology and Downscaled Climate Projections  

This notebook trains and evaluates the 5 academic models:
- 3 Traditional Algorithms: Random Forest, XGBoost, SVR
- 2 Hybrid Architectures: CNN-LSTM, LSTM Autoencoder + Random Forest

And exports ready-to-deploy **Artifact Bundles** for consumption in **Next.js + FastAPI**.

In [ ]:
# 1. Clone repository and install dependencies
!pip install -q xgboost reportlab plotly scikit-learn tensorflow

import os, sys
if os.path.exists('agro-digital-twin-st'):
    %cd agro-digital-twin-st

sys.path.insert(0, os.path.abspath('.'))

In [ ]:
# 2. Inspect Hardware Accelerators (GPU / CPU)
from src.core.hardware import detect_compute_device
hw = detect_compute_device()
print('Compute Device Mode:', hw['device_mode'])
print('TensorFlow Version:', hw['tensorflow']['version'])
print('GPU Available:', hw['tensorflow']['gpu_available'])

In [ ]:
# 3. Load Multi-Scale Eco-Hydrological Dataset
from src.core.dataset_generator import get_dataset
df = get_dataset()
print(f'Loaded dataset: {df.shape[0]} records, {df.shape[1]} columns')
print(f'Provenance: {df["data_provenance"].iloc[0]}')
df.head()

In [ ]:
# 4. Run Leak-Free Multi-Scale Training Pipeline
from src.core.training.trainer import MultiScaleTrainer

trainer = MultiScaleTrainer(
    target_name='monthly_runoff_mm',
    learning_mode='direct',  # or 'residual'
    validation_strategy='temporal',
    fast_dev_mode=False
)

results = trainer.train(
    df,
    progress_callback=lambda pct, msg: print(f'[{int(pct*100)}%] {msg}')
)

print('\n🏆 Champion Model:', results['champion_model_name'])
print('Champion Metrics:', results['champion_metrics'])

In [ ]:
# 5. Test Standalone Inference with ModelBundle (Zero Streamlit dependency)
from src.core.inference import ModelBundle

bundle = ModelBundle.load('artifacts/monthly_runoff_mm/champion')
sample_input = {
    'precip_mm': 110.0,
    'temp_mean_c': 22.5,
    'solar_radiation': 21.0,
    'soil_moisture': 32.0,
    'infiltration_mm': 60.0,
    'lai': 4.5,
    'root_depth_m': 1.3,
    'transpiration_mm': 68.0,
    'water_stress': 0.06,
    'et_mm': 90.0
}

prediction = bundle.predict(sample_input)
print('Prediction output:', prediction)

In [ ]:
# 6. Download Artifact Bundle for FastAPI
import zipfile
from google.colab import files

zip_name = 'artifacts_bundle.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, f_list in os.walk('artifacts'):
        for f in f_list:
            p = os.path.join(root, f)
            zipf.write(p, p)

print(f'Archive created: {zip_name}. Downloading...')
try:
    files.download(zip_name)
except Exception as e:
    print('Download ready at:', zip_name)